In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# DUMMY SUBMISSION:

In [2]:
# import pandas as pd

# submission_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# submission_df.to_csv("submission.csv",index=False)

# IMPORTING ALL DEPENDENCIES:

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re


# LOADING ALL DATASETS:

In [4]:
train_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
submission_df=pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

# EXPLONATORY DATA ANALYSIS:

#  DISPLAY FIRST FIVE ROWS OF TRAIN_DF:

In [5]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [7]:
print("#" * 50)
print(f"TRAIN_DF SHAPE: {train_df.shape}")
print(f"TEST_DF SHAPE: {test_df.shape}")

print("#" * 50)
print(f"TRAIN_DF COLUMNS: {train_df.columns}")
print(f"TEST_DF COLUMNS: {test_df.columns}")

print("#" * 50)
print(f"MISSING VALUE IN TRAIN_DF:{train_df.isnull().sum()}")
print(f"MISSING VALUE IN TEST_DF:{test_df.isnull().sum()}")

print("#" * 50)
print(f"DISTRIBUTION OF TARGET COLUMN IN TRAIN DF:{train_df["answer"].value_counts()}")


##################################################
TRAIN_DF SHAPE: (2000, 8)
TEST_DF SHAPE: (500, 7)
##################################################
TRAIN_DF COLUMNS: Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'], dtype='object')
TEST_DF COLUMNS: Index(['id', 'prompt', 'A', 'B', 'C', 'D', 'E'], dtype='object')
##################################################
MISSING VALUE IN TRAIN_DF:id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64
MISSING VALUE IN TEST_DF:id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64
##################################################
DISTRIBUTION OF TARGET COLUMN IN TRAIN DF:answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64


In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


# HANDLING MISSING DATA:

In [9]:
columns = ["prompt", "A", "B", "C", "D", "E"]
for column in columns:
    train_df[column] = train_df[column].fillna("")
    if column in test_df.columns:
        test_df[column] = test_df[column].fillna("")

# TEXT CLEANING AND TOKENIZATION:

In [10]:
def clean_and_tokenize(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    tokens = text.split()
    return tokens


for col in train_df.columns:
    train_df[col]=train_df[col].apply(lambda x:" ".join(clean_and_tokenize(x)))
    if col in test_df.columns:
        test_df[col]=test_df[col].apply(lambda x:" ".join(clean_and_tokenize(x)))

In [11]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,b
1,2,what is acceleratorbased lightion fusion,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,acceleratorbased lightion fusion is a techniqu...,a
2,3,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing,c
3,4,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...,b
4,5,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...,a


In [12]:
test_df.head()

,id,prompt,A,B,C,D,E
0,1,pick the best possible answer what is the rela...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...,for every eigenstate of one hamiltonian its pa...
1,2,what is the estimated redshift of ceers93316 a...,approximately z 60 corresponding to 1 billion ...,approximately z 167 corresponding to 2358 mill...,approximately z 30 corresponding to 5 billion ...,approximately z 100 corresponding to 13 billio...,approximately z 130 corresponding to 30 billio...
2,3,pick the best possible answer what is the reas...,the sun appears yellowish due to a reflection ...,the longer wavelengths of light such as red an...,the sun appears yellowish due to the scatterin...,the sun emits a yellow light due to its own sp...,the atmosphere absorbs the shorter wavelengths...
3,4,what is the significance of the redshiftdistan...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...,observations of the redshiftdistance relations...
4,5,what is the landaulifshitzgilbert equation use...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...,the landaulifshitzgilbert equation is a differ...


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Build vocabulary on all text elements across the train set
corpus = (
    train_df["prompt"].tolist() +
    train_df["A"].tolist() +
    train_df["B"].tolist() +
    train_df["C"].tolist() +
    train_df["D"].tolist() +
    train_df["E"].tolist()
)

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)



=== TF-IDF Embeddings ===
Vocabulary size: 3055 features.
